# Titanic competition with TensorFlow Decision Forests

This notebook will take you through the steps needed to train a baseline Gradient Boosted Trees Model using TensorFlow Decision Forests and creating a submission on the Titanic competition. 

This notebook shows:

1. How to do some basic pre-processing. For example, the passenger names will be tokenized, and ticket names will be splitted in parts.
1. How to train a Gradient Boosted Trees (GBT) with default parameters
1. How to train a GBT with improved default parameters
1. How to tune the parameters of a GBTs
1. How to train and ensemble many GBTs

# Imports dependencies

In [18]:
import numpy as np
import pandas as pd
import os

import tensorflow as tf
import tensorflow_decision_forests as tfdf

print(f"Found TF-DF {tfdf.__version__}")

Found TF-DF 1.2.0


# Load dataset

In [19]:
train_df = pd.read_csv("/kaggle/input/titanic/train.csv")
serving_df = pd.read_csv("/kaggle/input/titanic/test.csv")

train_df.head(10)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C


# Prepare dataset

We will apply the following transformations on the dataset.

1. Tokenize the names. For example, "Braund, Mr. Owen Harris" will become ["Braund", "Mr.", "Owen", "Harris"].
2. Extract any prefix in the ticket. For example ticket "STON/O2. 3101282" will become "STON/O2." and 3101282.

In [92]:
def preprocess(df):
    df = df.copy()
    
    def normalize_name(x):
        return " ".join([v.strip(",()[].\"'") for v in x.split(" ")])
    
    def ticket_number(x):
        return x.split(" ")[-1]
        
    def ticket_item(x):
        items = x.split(" ")
        if len(items) == 1:
            return "NONE"
        return "_".join(items[0:-1])
    
    def get_title(name):
        title = name.split(",")[1].split(".")[0].strip()
        rare_titles = ["Lady","Countess","Capt","Col","Don","Dr",
                       "Major","Rev","Sir","Jonkheer","Dona"]
        if title in rare_titles:
            return "Rare"
        mapping = {"Mlle":"Miss","Ms":"Miss","Mme":"Mrs"}
        return mapping.get(title, title)
    
    df["Title"] = df["Name"].apply(get_title)
    
    # FamilySize: +1でバグ修正済み（本人を含む人数）
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    #df["IsAlone"] = (df["FamilySize"] == 1).astype(int)
    
    df["Sex_Pclass"] = df["Sex"] + "_" + df["Pclass"].astype(str)
    
    #df["Deck"] = df["Cabin"].astype(str).str[0]
    #df["HasCabin"] = df["Cabin"].notna().astype(int)
    
    #df["FarePerPerson"] = df["Fare"] / df["FamilySize"]
    
    #df["IsChild"] = (df["Age"] < 16).astype(int)
    
    #df["NameLength"] = df["Name"].apply(len)
    
    # Age補完は行わない。TF-DFのネイティブ欠損値処理に任せる
    
    df["Name"] = df["Name"].apply(normalize_name)
    df["Ticket_number"] = df["Ticket"].apply(ticket_number)
    #df["Ticket_item"] = df["Ticket"].apply(ticket_item)
    
    # チケットのグループサイズ（trainとtestで別々に計算せず、まとめて計算する場合は
    # preprocess()の外でconcatしてから計算する必要あり。ここでは単体df内で計算。）
    #ticket_counts = df["Ticket"].value_counts()
    #df["TicketGroupSize"] = df["Ticket"].map(ticket_counts)
    
    return df

    
preprocessed_train_df = preprocess(train_df)
preprocessed_serving_df = preprocess(serving_df)

preprocessed_train_df.head(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Title,FamilySize,Sex_Pclass,Ticket_number
0,1,0,3,Braund Mr Owen Harris,male,22.0,1,0,A/5 21171,7.2500,NaN,S,Mr,2,male_3,21171
1,2,1,1,Cumings Mrs John Bradley Florence Briggs Thayer,female,38.0,1,0,PC 17599,71.2833,C85,C,Mrs,2,female_1,17599
2,3,1,3,Heikkinen Miss Laina,female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,Miss,1,female_3,3101282
3,4,1,1,Futrelle Mrs Jacques Heath Lily May Peel,female,35.0,1,0,113803,53.1000,C123,S,Mrs,2,female_1,113803
4,5,0,3,Allen Mr William Henry,male,35.0,0,0,373450,8.0500,NaN,S,Mr,1,male_3,373450


Let's keep the list of the input features of the model. Notably, we don't want to train our model on the "PassengerId" and "Ticket" features.

In [93]:
input_features = list(preprocessed_train_df.columns)
input_features.remove("Ticket")
input_features.remove("PassengerId")
input_features.remove("Survived")
#input_features.remove("Ticket_number")
input_features.remove("Embarked")     
input_features.remove("Cabin")  
input_features.remove("Sex")  
input_features.remove("Pclass")  
input_features.remove("SibSp")  
input_features.remove("Parch")  
input_features.remove("Fare") 

print(f"Input features: {input_features}")

Input features: ['Name', 'Age', 'Title', 'FamilySize', 'Sex_Pclass', 'Ticket_number']


# Convert Pandas dataset to TensorFlow Dataset

In [94]:
def tokenize_names(features, labels=None):
    """Divite the names into tokens. TF-DF can consume text tokens natively."""
    features["Name"] =  tf.strings.split(features["Name"])
    return features, labels

train_ds = tfdf.keras.pd_dataframe_to_tf_dataset(preprocessed_train_df,label="Survived").map(tokenize_names)
serving_ds = tfdf.keras.pd_dataframe_to_tf_dataset(preprocessed_serving_df).map(tokenize_names)

# Train model with default parameters

### Train model

First, we are training a GradientBoostedTreesModel model with the default parameters.

In [86]:
model = tfdf.keras.GradientBoostedTreesModel(
    verbose=0, # Very few logs
    features=[tfdf.keras.FeatureUsage(name=n) for n in input_features],
    exclude_non_specified_features=True, # Only use the features in "features"
    random_seed=1234,
)
model.fit(train_ds)

self_evaluation = model.make_inspector().evaluation()
print(f"Accuracy: {self_evaluation.accuracy} Loss:{self_evaluation.loss}")

[INFO 2026-07-05T06:48:17.337164708+00:00 kernel.cc:1214] Loading model from path /tmp/tmpay_3cajy/model/ with prefix ad43b94456974875
[INFO 2026-07-05T06:48:17.344243481+00:00 abstract_model.cc:1311] Engine "GradientBoostedTreesQuickScorerExtended" built
[INFO 2026-07-05T06:48:17.344283138+00:00 kernel.cc:1046] Use fast generic engine


Accuracy: 0.8478260636329651 Loss:0.826004683971405


# Train model with improved default parameters

Now you'll use some specific parameters when creating the GBT model

In [87]:
model = tfdf.keras.GradientBoostedTreesModel(
    verbose=0, # Very few logs
    features=[tfdf.keras.FeatureUsage(name=n) for n in input_features],
    exclude_non_specified_features=True, # Only use the features in "features"
    
    #num_trees=2000,
    
    # Only for GBT.
    # A bit slower, but great to understand the model.
    # compute_permutation_variable_importance=True,
    
    # Change the default hyper-parameters
    # hyperparameter_template="benchmark_rank1@v1",
    
    #num_trees=1000,
    #tuner=tuner
    
    min_examples=1,
    categorical_algorithm="RANDOM",
    #max_depth=4,
    shrinkage=0.05,
    #num_candidate_attributes_ratio=0.2,
    split_axis="SPARSE_OBLIQUE",
    sparse_oblique_normalization="MIN_MAX",
    sparse_oblique_num_projections_exponent=2.0,
    num_trees=2000,
    #validation_ratio=0.0,
    random_seed=1234,
    
)
model.fit(train_ds)

self_evaluation = model.make_inspector().evaluation()
print(f"Accuracy: {self_evaluation.accuracy} Loss:{self_evaluation.loss}")

[INFO 2026-07-05T06:48:23.999918396+00:00 kernel.cc:1214] Loading model from path /tmp/tmpoh3d250d/model/ with prefix 597a927f016147a6
[INFO 2026-07-05T06:48:24.011844239+00:00 decision_forest.cc:661] Model loaded with 57 root(s), 2689 node(s), and 6 input feature(s).
[INFO 2026-07-05T06:48:24.011892885+00:00 kernel.cc:1046] Use fast generic engine


Accuracy: 0.8369565010070801 Loss:0.8859865069389343


Let's look at the model and you can also notice the information about variable importance that the model figured out

In [88]:
model.summary()

Model: "gradient_boosted_trees_model_138"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
Total params: 1
Trainable params: 0
Non-trainable params: 1
_________________________________________________________________
Type: "GRADIENT_BOOSTED_TREES"
Task: CLASSIFICATION
Label: "__LABEL"

Input Features (6):
	Age
	FamilySize
	Name
	Sex_Pclass
	Ticket_number
	Title

No weights

Variable Importance: INV_MEAN_MIN_DEPTH:
    1.    "Sex_Pclass"  0.699823 ################
    2.         "Title"  0.277071 ##
    3.           "Age"  0.273924 ##
    4.          "Name"  0.252447 #
    5. "Ticket_number"  0.200686 

Variable Importance: NUM_AS_ROOT:
    1. "Sex_Pclass" 45.000000 ################
    2.      "Title"  9.000000 ##
    3.       "Name"  3.000000 

Variable Importance: NUM_NODES:
    1.           "Age" 718.000000 ################
    2.          "Name" 267.000000 ####
    3.    "Sex_Pclass" 151.000000 #
   

In [89]:
model_with_importance = tfdf.keras.GradientBoostedTreesModel(
    verbose=0,
    features=[tfdf.keras.FeatureUsage(name=n) for n in input_features],
    exclude_non_specified_features=True,
    compute_permutation_variable_importance=True,
    random_seed=1234,
)
model_with_importance.fit(train_ds)
model_with_importance.make_inspector().variable_importances()

[INFO 2026-07-05T06:48:42.775869531+00:00 kernel.cc:1214] Loading model from path /tmp/tmpxo8onbe2/model/ with prefix 87bde94b76f64e19
[INFO 2026-07-05T06:48:42.78266812+00:00 abstract_model.cc:1311] Engine "GradientBoostedTreesQuickScorerExtended" built
[INFO 2026-07-05T06:48:42.782713378+00:00 kernel.cc:1046] Use fast generic engine


{'MEAN_DECREASE_IN_PRAUC_2_VS_OTHERS': [("Sex_Pclass" (4; #3),
   0.24467950235026548),
  ("FamilySize" (1; #1), 0.029470134853345042),
  ("Name" (5; #2), 0.015567992120061103),
  ("Title" (4; #5), 0.012826218588942284),
  ("Age" (1; #0), 0.008675345577893623),
  ("Ticket_number" (4; #4), 0.005742475445284612)],
 'SUM_SCORE': [("Sex_Pclass" (4; #3), 344.10882171344383),
  ("Title" (4; #5), 81.97851053329032),
  ("Name" (5; #2), 80.00543233748495),
  ("Age" (1; #0), 65.52424416274152),
  ("FamilySize" (1; #1), 64.8640637271501),
  ("Ticket_number" (4; #4), 20.853206141968826)],
 'INV_MEAN_MIN_DEPTH': [("Sex_Pclass" (4; #3), 0.4449038161395111),
  ("Name" (5; #2), 0.31708006427826185),
  ("Title" (4; #5), 0.25904490476742253),
  ("Age" (1; #0), 0.25049103795346045),
  ("FamilySize" (1; #1), 0.2080740820122068),
  ("Ticket_number" (4; #4), 0.20742494159075964)],
 'MEAN_DECREASE_IN_AP_2_VS_OTHERS': [("Sex_Pclass" (4; #3),
   0.25032251501670966),
  ("FamilySize" (1; #1), 0.0299137108513860

# Make predictions

In [90]:
def prediction_to_kaggle_format(model, threshold=0.5):
    proba_survive = model.predict(serving_ds, verbose=0)[:,0]
    return pd.DataFrame({
        "PassengerId": serving_df["PassengerId"],
        "Survived": (proba_survive >= threshold).astype(int)
    })

def make_submission(kaggle_predictions):
    path="/kaggle/working/submission.csv"
    kaggle_predictions.to_csv(path, index=False)
    print(f"Submission exported to {path}")
    
kaggle_predictions = prediction_to_kaggle_format(model)
make_submission(kaggle_predictions)
!head /kaggle/working/submission.csv

Submission exported to /kaggle/working/submission.csv
PassengerId,Survived
892,0
893,0
894,0
895,0
896,1
897,0
898,0
899,0
900,1


# Training a model with hyperparameter tunning

Hyper-parameter tuning is enabled by specifying the tuner constructor argument of the model. The tuner object contains all the configuration of the tuner (search space, optimizer, trial and objective).


In [91]:
tuner = tfdf.tuner.RandomSearch(num_trials=1000)
tuner.choice("min_examples", [2, 5, 7, 10])
tuner.choice("categorical_algorithm", ["CART", "RANDOM"])

local_search_space = tuner.choice("growing_strategy", ["LOCAL"])
local_search_space.choice("max_depth", [3, 4, 5, 6, 8])

global_search_space = tuner.choice("growing_strategy", ["BEST_FIRST_GLOBAL"], merge=True)
global_search_space.choice("max_num_nodes", [16, 32, 64, 128, 256])

#tuner.choice("use_hessian_gain", [True, False])
tuner.choice("shrinkage", [0.02, 0.05, 0.10, 0.15])
tuner.choice("num_candidate_attributes_ratio", [0.2, 0.5, 0.9, 1.0])


tuner.choice("split_axis", ["AXIS_ALIGNED"])
oblique_space = tuner.choice("split_axis", ["SPARSE_OBLIQUE"], merge=True)
oblique_space.choice("sparse_oblique_normalization",
                     ["NONE", "STANDARD_DEVIATION", "MIN_MAX"])
oblique_space.choice("sparse_oblique_weights", ["BINARY", "CONTINUOUS"])
oblique_space.choice("sparse_oblique_num_projections_exponent", [1.0, 1.5])

# Tune the model. Notice the `tuner=tuner`.
tuned_model = tfdf.keras.GradientBoostedTreesModel(tuner=tuner)
tuned_model.fit(train_ds, verbose=0)

tuned_self_evaluation = tuned_model.make_inspector().evaluation()
print(f"Accuracy: {tuned_self_evaluation.accuracy} Loss:{tuned_self_evaluation.loss}")

Use /tmp/tmpg4y8mxqc as temporary training directory


[INFO 2026-07-05T06:51:02.803696123+00:00 kernel.cc:1214] Loading model from path /tmp/tmpg4y8mxqc/model/ with prefix ce56afbde4bb4fda
[INFO 2026-07-05T06:51:02.824934619+00:00 decision_forest.cc:661] Model loaded with 32 root(s), 2134 node(s), and 15 input feature(s).
[INFO 2026-07-05T06:51:02.824979662+00:00 abstract_model.cc:1311] Engine "GradientBoostedTreesGeneric" built
[INFO 2026-07-05T06:51:02.82500603+00:00 kernel.cc:1046] Use fast generic engine


Accuracy: 0.8767123222351074 Loss:0.6400823593139648


In the last line in the cell above, you can see the accuracy is higher than previously with default parameters and parameters set by hand.

This is the main idea behing hyperparameter tuning.

For more information you can follow this tutorial: [Automated hyper-parameter tuning](https://www.tensorflow.org/decision_forests/tutorials/automatic_tuning_colab)

# Making an ensemble

Here you'll create 100 models with different seeds and combine their results

This approach removes a little bit the random aspects related to creating ML models

In the GBT creation is used the `honest` parameter. It will use different training examples to infer the structure and the leaf values. This regularization technique trades examples for bias estimates.

In [95]:
predictions = None
num_predictions = 0

for i in range(100):
    print(f"i:{i}")
    # Possible models: GradientBoostedTreesModel or RandomForestModel
    model = tfdf.keras.GradientBoostedTreesModel(
        verbose=0, # Very few logs
        features=[tfdf.keras.FeatureUsage(name=n) for n in input_features],
        exclude_non_specified_features=True, # Only use the features in "features"

        #min_examples=1,
        #categorical_algorithm="RANDOM",
        ##max_depth=4,
        #shrinkage=0.05,
        ##num_candidate_attributes_ratio=0.2,
        #split_axis="SPARSE_OBLIQUE",
        #sparse_oblique_normalization="MIN_MAX",
        #sparse_oblique_num_projections_exponent=2.0,
        #num_trees=2000,
        ##validation_ratio=0.0,
        random_seed=i,
        honest=True,
    )
    model.fit(train_ds)
    
    sub_predictions = model.predict(serving_ds, verbose=0)[:,0]
    if predictions is None:
        predictions = sub_predictions
    else:
        predictions += sub_predictions
    num_predictions += 1

predictions/=num_predictions

kaggle_predictions = pd.DataFrame({
        "PassengerId": serving_df["PassengerId"],
        "Survived": (predictions >= 0.5).astype(int)
    })

make_submission(kaggle_predictions)

i:0


[INFO 2026-07-05T06:55:52.710841859+00:00 kernel.cc:1214] Loading model from path /tmp/tmph5_4apyk/model/ with prefix b2f2cabaedde4fed
[INFO 2026-07-05T06:55:52.714649124+00:00 abstract_model.cc:1311] Engine "GradientBoostedTreesQuickScorerExtended" built
[INFO 2026-07-05T06:55:52.71469022+00:00 kernel.cc:1046] Use fast generic engine


i:1


[INFO 2026-07-05T06:55:53.807432154+00:00 kernel.cc:1214] Loading model from path /tmp/tmp90yy906r/model/ with prefix 92b06f88402642d2
[INFO 2026-07-05T06:55:53.817236294+00:00 kernel.cc:1046] Use fast generic engine


i:2


...

i:98


[INFO 2026-07-05T06:57:36.252435913+00:00 kernel.cc:1214] Loading model from path /tmp/tmpi1de4fbw/model/ with prefix d015738100084a1d
[INFO 2026-07-05T06:57:36.256864855+00:00 kernel.cc:1046] Use fast generic engine


i:99


[INFO 2026-07-05T06:57:37.207489632+00:00 kernel.cc:1214] Loading model from path /tmp/tmpspjab87i/model/ with prefix 4366637791314d82
[INFO 2026-07-05T06:57:37.214352114+00:00 kernel.cc:1046] Use fast generic engine


Submission exported to /kaggle/working/submission.csv
